# SELFIE on K2-V2 — Writer / Editor Agents

This notebook runs a [LangGraph](https://github.com/langchain-ai/langgraph) pipeline with two K2-V2-Instruct agents (writer and editor) and applies **SELFIE** (Self-Explanation of LLM Internal Representations) to probe their hidden states.

### Three experiments
| # | Name | What it does |
|---|------|--------------|
| 1 | `selfie_writer` | SELFIE on the writer's own output tokens — what does the model "think" each token means at different layers? |
| 2 | `injected_editor` | Replace the draft text with the writer's **hidden states** directly injected into the editor's residual stream — does the editor still understand? |
| 3 | `selfie_editor` | SELFIE on the editor's hidden states *at the draft span* — how does the editor internally represent the writer's text? |

### Hardware
K2-V2-Instruct is a **70B** model. bf16 needs ~140 GB VRAM.  
- **Multi-GPU**: `device_map="auto"` (default) shards across all visible GPUs  
- **Single 48 GB GPU**: use the 4-bit block below instead

## 0 — Install

In [ ]:
# Install the package from GitHub (pulls in all ML deps automatically)
# Replace the URL with your fork if needed.
!pip install -q git+https://github.com/YOUR_USERNAME/reagent.git

# Alternatively, if you cloned the repo and are running from inside it:
# !pip install -q -e ..

## 1 — Load K2-V2-Instruct

First run downloads ~140 GB of weights to `~/.cache/huggingface/`. Give it a few minutes.

In [ ]:
import time
import torch
import pandas as pd

from selfie_k2v2 import K2V2Backend, make_graph

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [ ]:
# ── Standard bf16 load (needs ~140 GB VRAM across GPUs) ──────────────────────
t0 = time.time()
backend = K2V2Backend(
    model_name="LLM360/K2-V2-Instruct",
    dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Model ready in {time.time() - t0:.1f}s")
print(f"  hidden_size = {backend.hidden_size}")
print(f"  num_layers  = {backend.num_layers}")
print(f"  device      = {backend.device}")

In [ ]:
# ── 4-bit alternative (fits on a single 48 GB GPU) ───────────────────────────
# Uncomment this block and skip the cell above if you're on a smaller GPU.

# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
#
# bnb = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_quant_type="nf4",
# )
# backend = K2V2Backend.__new__(K2V2Backend)
# backend.tokenizer = AutoTokenizer.from_pretrained("LLM360/K2-V2-Instruct")
# backend.model = AutoModelForCausalLM.from_pretrained(
#     "LLM360/K2-V2-Instruct", device_map="auto", quantization_config=bnb
# )
# backend.model.eval()
# backend.device = next(backend.model.parameters()).device
# backend.hidden_size = backend.model.config.hidden_size
# backend.num_layers = len(backend.model.model.layers)

## 2 — Sanity check: plain generation

Run a tiny prompt to confirm the model loads and decodes correctly before committing to the full graph.

In [ ]:
t0 = time.time()
ids = backend.build_chat_prompt(
    system="You are K2, a helpful assistant.",
    user="Say hi in 5 words.",
    reasoning_effort="medium",
)
r = backend.generate(ids, max_new_tokens=20, capture_hidden=False)
print(f"[{time.time() - t0:.1f}s]  {r.output_text!r}")

assert r.output_text.strip(), (
    "Empty output — something is wrong with the model load or chat template. "
    "Check your GPU memory and transformers version."
)

## 3 — Build the graph and run all experiments

```
writer → editor → selfie_writer   (Exp 1)
                → selfie_editor   (Exp 3)
                → injected_editor (Exp 2) → END
```

Change `TASK` to whatever you like. Keeping token caps small makes the SELFIE passes fast.

In [ ]:
TASK = "Describe a golden retriever in 3 sentences."
MAX_WRITER_TOKENS = 40
MAX_EDITOR_TOKENS = 60

graph = make_graph(backend, max_writer_tokens=MAX_WRITER_TOKENS, max_editor_tokens=MAX_EDITOR_TOKENS)

print(f"Running graph on task: {TASK!r}")
t0 = time.time()
final = graph.invoke({"task": TASK})
print(f"\nDone in {time.time() - t0:.1f}s")

## 4 — Agent outputs (text level)

In [ ]:
print("── WRITER OUTPUT " + "─" * 55)
print(final["writer_output_text"])

print("\n── EDITOR VERDICT — normal (sees writer's text) " + "─" * 24)
print(final["editor_verdict"])

print("\n── EDITOR VERDICT — injected (hidden states at placeholders) " + "─" * 11)
print(final["editor_injected_verdict"])

print()
print("If the two verdicts are similar, the writer's final hidden states carry")
print("roughly the same semantic content as its surface tokens.")
print("If injected is garbled, try inject_layer=1 or a middle layer in")
print("agents_graph.injected_editor_node.")

## 5 — Experiment 1: SELFIE on the writer's hidden states

Each row = one `(layer, token)` pair from the writer's output, interpreted by patching the SELFIE interpretation prompt at that layer.

In [ ]:
df_writer = final["selfie_writer"]
print(f"{len(df_writer)} interpretations across {df_writer['layer'].nunique()} probe layers\n")
df_writer

## 6 — Experiment 3: SELFIE on the editor's hidden states at the draft span

Same idea but now we look at the *editor's* representation of the writer's tokens — the same surface tokens read through a different model run.

In [ ]:
df_editor = final["selfie_editor_on_draft"]
print(f"{len(df_editor)} interpretations across {df_editor['layer'].nunique()} probe layers\n")
df_editor

## 7 — Side-by-side: writer vs editor representations

Align both SELFIE tables by `(layer, rel_pos)` where `rel_pos` is the offset within the writer's output span.  
Direct comparison of how the two agents internally represent the same token at the same layer depth.

In [ ]:
wr = final["writer_result"]
writer_first = wr.prompt_len
draft_first  = final["editor_draft_start"]

w = df_writer.copy()
w["rel_pos"] = w["token_idx"] - writer_first
w = w.rename(columns={"token": "writer_tok", "interpretation": "writer_interp"})

e = df_editor.copy()
e["rel_pos"] = e["token_idx"] - draft_first
e = e.rename(columns={"token": "editor_tok", "interpretation": "editor_interp"})

merged = w.merge(
    e[["layer", "rel_pos", "editor_tok", "editor_interp"]],
    on=["layer", "rel_pos"],
    how="inner",
)[["layer", "rel_pos", "writer_tok", "editor_tok", "writer_interp", "editor_interp"]]
merged = merged.sort_values(["layer", "rel_pos"]).reset_index(drop=True)
merged

## 8 — Save outputs (optional)

In [ ]:
import os

out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

df_writer.to_csv(f"{out_dir}/selfie_writer.csv", index=False)
df_editor.to_csv(f"{out_dir}/selfie_editor_on_draft.csv", index=False)
merged.to_csv(f"{out_dir}/merged_side_by_side.csv", index=False)

with open(f"{out_dir}/texts.txt", "w") as f:
    f.write("=== writer_output_text ===\n")
    f.write(final["writer_output_text"] + "\n\n")
    f.write("=== editor_verdict (normal) ===\n")
    f.write(final["editor_verdict"] + "\n\n")
    f.write("=== editor_injected_verdict ===\n")
    f.write(final["editor_injected_verdict"] + "\n")

print(f"Saved to {out_dir}/")
print("  selfie_writer.csv")
print("  selfie_editor_on_draft.csv")
print("  merged_side_by_side.csv")
print("  texts.txt")